# 03 — Provider flags

Join the ODS trust list onto the RTT panel: sector flag, trust name, ICB, and per-provider stability columns. Deliverable: `data/processed/rtt_panel_with_provider_flags.csv`.

In [1]:
import pandas as pd                          # the table library: joins, groupbys, csv io
from pathlib import Path                     # path objects that work on Windows and Linux

PROC = Path("../data/processed")             # outputs of 02_eda_dataset live here
panel = pd.read_csv(PROC / "rtt_incomplete_panel.csv", parse_dates=["month"])
ods = pd.read_csv("../data/reference/trust_icb.csv")   # ODS RO197 trust list from 01_fetch

print(panel.shape, "-", panel["provider_code"].nunique(), "providers")
print(ods.shape, "ods trusts")

(343441, 8) - 651 providers
(544, 5) ods trusts


## Sector flag

A provider is *public* exactly when its code appears in the ODS RO197 trust list — codes not in that list are independent-sector providers (private hospitals, clinics). No manual lists needed.

In [2]:
# Goal: attach trust name / status / ICB to every panel row and flag the sector.
panel = panel.merge(ods.rename(columns={"trust_code": "provider_code"}),
                    on="provider_code", how="left")     # left join: independents get NaN
panel["is_public"] = panel["trust_name"].notna()

pub = panel.groupby("is_public")["total_waiting"].sum()   # denominator: all patient-months
print(panel.drop_duplicates("provider_code")["is_public"].value_counts())
print("share of waiting list held by public trusts: %.1f%%" % (pub[True] / pub.sum() * 100))   # expect ~94.5

is_public
False    458
True     193
Name: count, dtype: int64
share of waiting list held by public trusts: 94.5%


## Stability columns and flags

Thresholds are declared choices (2026-07-29), not tuned: a modeling client needs 4+ years of history and must still be reporting in the final data month; an allocatable provider is public with 2+ years of history.

In [3]:
# Goal: per-provider history length - the basis for choosing stable modeling clients.
stats = panel.groupby("provider_code")["month"].agg(
    n_months_active="nunique", first_month="min", last_month="max").reset_index()
panel = panel.merge(stats, on="provider_code")

print(stats["n_months_active"].describe())

count    651.000000
mean      65.221198
std       36.323888
min        1.000000
25%       30.000000
50%       89.000000
75%       96.000000
max       96.000000
Name: n_months_active, dtype: float64


In [4]:
FINAL_MONTH = panel["month"].max()           # 2026-03-01, data-derived - no hard-coded date

panel["fl_client"] = (panel["n_months_active"] >= 48) & (panel["last_month"] == FINAL_MONTH)
panel["fl_allocatable"] = panel["is_public"] & (panel["n_months_active"] >= 24)

flags = panel.drop_duplicates("provider_code")           # count providers, not rows
print("fl_client     :", int(flags["fl_client"].sum()), "of", len(flags), "providers")
print("fl_allocatable:", int(flags["fl_allocatable"].sum()), "of", len(flags), "providers")

fl_client     : 392 of 651 providers
fl_allocatable: 182 of 651 providers


In [5]:
panel.to_csv(PROC / "rtt_panel_with_provider_flags.csv", index=False)
print(panel.shape, "->", PROC / "rtt_panel_with_provider_flags.csv")
panel.head()

(343441, 18) -> ..\data\processed\rtt_panel_with_provider_flags.csv


,month,provider_code,tfunc_code,total_waiting,gt18,gt52,pct_gt18,pct_gt52,trust_name,status,icb_code,postcode,is_public,n_months_active,first_month,last_month,fl_client,fl_allocatable
0,2019-10-01,AAH,C_100,12,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False,96,2018-04-01,2026-03-01,True,False
1,2019-10-01,AAH,C_101,33,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False,96,2018-04-01,2026-03-01,True,False
2,2019-10-01,AAH,C_110,35,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False,96,2018-04-01,2026-03-01,True,False
3,2019-10-01,AAH,C_120,68,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False,96,2018-04-01,2026-03-01,True,False
4,2019-10-01,AAH,C_130,233,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,False,96,2018-04-01,2026-03-01,True,False
